In [0]:
# Import required PySpark functions
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
# 1. Load Data from Existing Databricks Table

# Read the retail dataset from the Databricks table
df = spark.table("demoworkspacejoby.default.retail_data_analytics")

# Preview the data
df.show(5)

# Check schema
df.printSchema()

+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
| 557112|    23243|SET OF TEA COFFEE...|       1|2011-06-16 16:31:00| 4.95|       NULL|United Kingdom|
| 557112|    23245|SET OF 3 REGENCY ...|       4|2011-06-16 16:31:00|10.79|       NULL|United Kingdom|
| 557112|    23251|VINTAGE RED ENAME...|       2|2011-06-16 16:31:00| 2.46|       NULL|United Kingdom|
| 557112|    23256|CHILDRENS CUTLERY...|       1|2011-06-16 16:31:00| 8.29|       NULL|United Kingdom|
| 557112|    23298|      SPOTTY BUNTING|       2|2011-06-16 16:31:00|10.79|       NULL|United Kingdom|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
only showing top 5 rows
root
 |-- Invoice: string (nullable = true)
 |-- 

In [0]:
# 2. Clean Base Data

# Remove duplicate rows
df_base = df.dropDuplicates()

# Remove rows where Customer ID is null
df_base = df_base.filter(col("Customer ID").isNotNull())

# Convert Customer ID to integer
df_base = df_base.withColumn(
    "Customer ID",
    col("Customer ID").cast("bigint")
)

# Remove rows where Price is less than or equal to 0
df_base = df_base.filter(col("Price") > 0)

# Convert InvoiceDate column to timestamp format
df_base = df_base.withColumn("InvoiceDate", to_timestamp(col("InvoiceDate")))

# Create Revenue column
df_base = df_base.withColumn("Revenue", col("Quantity") * col("Price"))

# Create cancellation flag
df_base = df_base.withColumn(
    "is_cancelled",
    when(col("Invoice").startswith("C"), 1).otherwise(0)
)

# Create sales-only dataframe
df_clean = df_base.filter(col("Quantity") > 0)

# Preview cleaned sales data
df_clean.show(5)

+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+-------+------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|Revenue|is_cancelled|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+-------+------------+
| 489434|    21871| SAVE THE PLANET MUG|      24|2009-12-01 07:45:00| 1.25|      13085|United Kingdom|   30.0|           0|
| 489436|    21754|HOME BUILDING BLO...|       3|2009-12-01 09:06:00| 5.95|      13078|United Kingdom|  17.85|           0|
| 489436|    22109|FULL ENGLISH BREA...|      16|2009-12-01 09:06:00| 3.39|      13078|United Kingdom|  54.24|           0|
| 489439|    21493| VINTAGE DESIGN G...|      12|2009-12-01 09:28:00| 0.85|      12682|        France|   10.2|           0|
| 489440|    22350|           CAT BOWL |       8|2009-12-01 09:43:00| 2.55|      18087|United Kingdom|   20.4|           0|
+-------

In [0]:
# 3. Create Sales-Only Data

# Keep only valid positive sales transactions
# This is used for revenue analysis and RFM analysis
df_sales = df_clean.filter(col("Quantity") > 0)

# Preview sales data
df_sales.show(5)

+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+-------+------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|Revenue|is_cancelled|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+-------+------------+
| 489434|    21871| SAVE THE PLANET MUG|      24|2009-12-01 07:45:00| 1.25|      13085|United Kingdom|   30.0|           0|
| 489436|    21754|HOME BUILDING BLO...|       3|2009-12-01 09:06:00| 5.95|      13078|United Kingdom|  17.85|           0|
| 489436|    22109|FULL ENGLISH BREA...|      16|2009-12-01 09:06:00| 3.39|      13078|United Kingdom|  54.24|           0|
| 489439|    21493| VINTAGE DESIGN G...|      12|2009-12-01 09:28:00| 0.85|      12682|        France|   10.2|           0|
| 489440|    22350|           CAT BOWL |       8|2009-12-01 09:43:00| 2.55|      18087|United Kingdom|   20.4|           0|
+-------

In [0]:
# 4. Monthly Placed and Canceled Orders

df_base = df.dropDuplicates()

df_base = df_base.filter(col("Customer ID").isNotNull())
df_base = df_base.filter(col("Price") > 0)

# Create YYYYMM column
df_monthly_orders = df_base.withColumn(
    "invoice_year_month",
    year(col("InvoiceDate")) * 100 + month(col("InvoiceDate"))
)

# Identify cancelled invoices
df_monthly_orders = df_monthly_orders.withColumn(
    "is_canceled",
    col("Invoice").startswith("C")
)

# Total distinct invoices
monthly_total_orders_df = (
    df_monthly_orders
    .groupBy("invoice_year_month")
    .agg(countDistinct("Invoice").alias("total_orders"))
)

# Cancelled distinct invoices
monthly_canceled_orders_df = (
    df_monthly_orders
    .filter(col("is_canceled") == True)
    .groupBy("invoice_year_month")
    .agg(countDistinct("Invoice").alias("canceled_orders"))
)

# Merge + calculate placed orders
monthly_orders_df = (
    monthly_total_orders_df
    .join(monthly_canceled_orders_df, on="invoice_year_month", how="left")
    .fillna(0, subset=["canceled_orders"])
    .withColumn("canceled_orders", col("canceled_orders").cast("int"))
    .withColumn("placed_orders", col("total_orders") - 2 * col("canceled_orders"))
    .orderBy("invoice_year_month")
)

monthly_orders_df.show()

+------------------+------------+---------------+-------------+
|invoice_year_month|total_orders|canceled_orders|placed_orders|
+------------------+------------+---------------+-------------+
|            200912|        1900|            388|         1124|
|            201001|        1296|            285|          726|
|            201002|        1333|            229|          875|
|            201003|        1907|            383|         1141|
|            201004|        1615|            286|         1043|
|            201005|        1768|            391|          986|
|            201006|        1833|            336|         1161|
|            201007|        1713|            332|         1049|
|            201008|        1547|            254|         1039|
|            201009|        2041|            352|         1337|
|            201010|        2586|            453|         1680|
|            201011|        3145|            558|         2029|
|            201012|        1708|       

In [0]:
# 5. Monthly Sales

# Step 1: Create YYYYMM column
df_monthly_sales = df_base.withColumn(
    "invoice_year_month",
    year(col("InvoiceDate")) * 100 + month(col("InvoiceDate"))
)

# Step 2: Filter valid sales
valid_sales_df = df_monthly_sales.filter(
    (~col("Invoice").startswith("C")) &
    (col("Quantity") > 0)
)

# Step 3: Calculate sales amount
valid_sales_df = valid_sales_df.withColumn(
    "sales_amount",
    col("Quantity") * col("Price")
)

# Step 4: Aggregate
monthly_sales_df = (
    valid_sales_df
    .groupBy("invoice_year_month")
    .agg(round(sum("sales_amount"), 2).alias("monthly_sales"))
    .orderBy("invoice_year_month")
)

monthly_sales_df.show()

+------------------+-------------+
|invoice_year_month|monthly_sales|
+------------------+-------------+
|            200912|    683504.01|
|            201001|    555802.67|
|            201002|    504558.96|
|            201003|    696978.47|
|            201004|     591982.0|
|            201005|    597833.38|
|            201006|    636371.13|
|            201007|    589736.17|
|            201008|     602224.6|
|            201009|    829013.95|
|            201010|   1033112.01|
|            201011|   1166460.02|
|            201012|    570422.73|
|            201101|    568101.31|
|            201102|    446084.92|
|            201103|    594081.76|
|            201104|    468374.33|
|            201105|    677355.15|
|            201106|    660046.05|
|            201107|     598962.9|
+------------------+-------------+
only showing top 20 rows


In [0]:
# 6. Monthly Sales Growth

# Prepare monthly sales data
monthly_sales = monthly_sales_df.withColumnRenamed("invoice_year_month", "yyyymm")

# Define window ordered by month
window_month = Window.orderBy("yyyymm")

# Calculate month-over-month sales growth
monthly_sales = (
    monthly_sales
    .withColumn(
        "sales_growth_pct",
        (col("monthly_sales") - lag("monthly_sales").over(window_month)) / lag("monthly_sales").over(window_month)
    )
)

monthly_sales.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------+-------------+--------------------+
|yyyymm|monthly_sales|    sales_growth_pct|
+------+-------------+--------------------+
|200912|    683504.01|                NULL|
|201001|    555802.67|-0.18683334425499562|
|201002|    504558.96|-0.09219766792412137|
|201003|    696978.47| 0.38136179367422185|
|201004|     591982.0|-0.15064521290019184|
|201005|    597833.38|0.009884388376673624|
|201006|    636371.13| 0.06446235906064662|
|201007|    589736.17|-0.07328264561593163|
|201008|     602224.6| 0.02117629990373481|
|201009|    829013.95|  0.3765859946604639|
|201010|   1033112.01| 0.24619375825943587|
|201011|   1166460.02| 0.12907410688217633|
|201012|    570422.73| -0.5109796133432846|
|201101|    568101.31|-0.00406964848683...|
|201102|    446084.92|-0.21477927942113012|
|201103|    594081.76|  0.3317683099442143|
|201104|    468374.33|-0.21159954481686155|
|201105|    677355.15| 0.44618333374504104|
|201106|    660046.05|-0.02555395053835492|
|201107|     598962.9|-0.0925437

In [0]:
# 7. Monthly Active Users

# Filter valid customer purchase records
valid_customers_df = df_base.filter(
    (~col("Invoice").startswith("C")) &
    (col("Quantity") > 0) &
    (col("Customer ID").isNotNull())
)

# Create year-month column
valid_customers_df = valid_customers_df.withColumn(
    "invoice_year_month",
    year(col("InvoiceDate")) * 100 + month(col("InvoiceDate"))
)

# Calculate monthly active users
monthly_active_users_df = (
    valid_customers_df
    .groupBy("invoice_year_month")
    .agg(countDistinct("Customer ID").alias("active_users"))
    .orderBy("invoice_year_month")
)

monthly_active_users_df.show()

+------------------+------------+
|invoice_year_month|active_users|
+------------------+------------+
|            200912|         955|
|            201001|         720|
|            201002|         772|
|            201003|        1057|
|            201004|         942|
|            201005|         966|
|            201006|        1041|
|            201007|         928|
|            201008|         911|
|            201009|        1145|
|            201010|        1497|
|            201011|        1607|
|            201012|         885|
|            201101|         741|
|            201102|         758|
|            201103|         974|
|            201104|         856|
|            201105|        1056|
|            201106|         991|
|            201107|         949|
+------------------+------------+
only showing top 20 rows


In [0]:
# 8. New and Existing Users

# Filter valid purchase records
user_tx_df = df_base.filter(
    (~col("Invoice").startswith("C")) &
    (col("Quantity") > 0) &
    (col("Customer ID").isNotNull())
)

# Create year-month column
user_tx_df = user_tx_df.withColumn(
    "invoice_year_month",
    year(col("InvoiceDate")) * 100 + month(col("InvoiceDate"))
)

# Rename column once
user_tx_df = user_tx_df.withColumnRenamed("Customer ID", "Customer_ID")

# First purchase month per customer
first_purchase_df = (
    user_tx_df
    .groupBy("Customer_ID")
    .agg(min("invoice_year_month").alias("first_purchase_month"))
)

# Join first purchase info
user_tx_enriched_df = user_tx_df.join(
    first_purchase_df,
    on="Customer_ID",
    how="left"
)

# Label user type
user_tx_enriched_df = user_tx_enriched_df.withColumn(
    "user_type",
    when(
        col("invoice_year_month") == col("first_purchase_month"),
        "new"
    ).otherwise("existing")
)

# Count unique users per month and type
monthly_user_type_df = (
    user_tx_enriched_df
    .groupBy("invoice_year_month", "user_type")
    .agg(countDistinct("Customer_ID").alias("user_count"))
)

# Pivot
monthly_user_pivot_df = (
    monthly_user_type_df
    .groupBy("invoice_year_month")
    .pivot("user_type", ["new", "existing"])
    .agg(first("user_count"))
    .fillna(0)
    .withColumnRenamed("new", "NewUserCount")
    .withColumnRenamed("existing", "ExUserCount")
    .orderBy("invoice_year_month")
)

monthly_user_pivot_df.show()

+------------------+------------+-----------+
|invoice_year_month|NewUserCount|ExUserCount|
+------------------+------------+-----------+
|            200912|         955|          0|
|            201001|         383|        337|
|            201002|         374|        398|
|            201003|         443|        614|
|            201004|         294|        648|
|            201005|         254|        712|
|            201006|         270|        771|
|            201007|         186|        742|
|            201008|         162|        749|
|            201009|         243|        902|
|            201010|         377|       1120|
|            201011|         325|       1282|
|            201012|          76|        809|
|            201101|          71|        670|
|            201102|         124|        634|
|            201103|         179|        795|
|            201104|         106|        750|
|            201105|         111|        945|
|            201106|         108| 

In [0]:
# 10. RFM Calculation

# Prepare dataframe
rfm_df = df_sales.withColumnRenamed("Customer ID", "Customer_ID")

# Calculate invoice amount per row
rfm_df = rfm_df.withColumn(
    "invoice_amount",
    col("Quantity") * col("Price")
)

# Define reference date (next day of max date)
reference_date = (
    rfm_df.select(date_add(to_date(max("InvoiceDate")), 1)).collect()[0][0]
)

# Aggregate RFM metrics
rfm_table = (
    rfm_df
    .groupBy("Customer_ID")
    .agg(
        datediff(lit(reference_date), to_date(max("InvoiceDate"))).alias("Recency"),
        countDistinct("Invoice").alias("Frequency"),
        round(sum("invoice_amount"), 2).alias("Monetary")
    )
)

rfm_table.show()

+-----------+-------+---------+---------+
|Customer_ID|Recency|Frequency| Monetary|
+-----------+-------+---------+---------+
|      12980|    158|       21| 16245.78|
|      12924|     89|       12|   2469.4|
|      15380|      9|       11|  3248.96|
|      15196|    433|        2|    313.7|
|      15920|    155|        9|   569.14|
|      17954|      6|       19|  3804.38|
|      16210|      2|       35| 33750.97|
|      17858|      6|       30| 12518.65|
|      13089|      3|      203|113416.91|
|      16641|    106|        2|    554.5|
|      18170|     34|        9|   2878.0|
|      15795|    150|        8|  1979.87|
|      17231|     13|       25|   7064.8|
|      14380|    683|        1|    48.96|
|      17625|     19|        8|   5978.9|
|      16779|      3|       53| 32809.03|
|      12500|     24|       19|  5955.21|
|      14030|     19|       20|   7521.8|
|      17287|     27|       21|  3826.86|
|      18092|      3|       18| 14693.72|
+-----------+-------+---------+---

In [0]:
# 11. RFM Scoring

# Define windows
window_r = Window.orderBy(col("Recency").asc())
window_f = Window.orderBy(col("Frequency").asc())
window_m = Window.orderBy(col("Monetary").asc())

# Recency score
rfm_metrics = rfm_table.withColumn(
    "R_score",
    6 - ntile(5).over(window_r)
)

# Frequency score
rfm_metrics = rfm_metrics.withColumn(
    "F_rank",
    row_number().over(Window.orderBy(col("Frequency")))
)

rfm_metrics = rfm_metrics.withColumn(
    "F_score",
    ntile(5).over(Window.orderBy(col("F_rank")))
)

# Monetary score
rfm_metrics = rfm_metrics.withColumn(
    "M_score",
    ntile(5).over(window_m)
)

# Convert scores to string for concatenation
rfm_metrics = rfm_metrics.withColumn("R_score", col("R_score").cast("string"))
rfm_metrics = rfm_metrics.withColumn("F_score", col("F_score").cast("string"))
rfm_metrics = rfm_metrics.withColumn("M_score", col("M_score").cast("string"))

# Create RFM score
rfm_metrics = rfm_metrics.withColumn(
    "RFM_score",
    concat(col("R_score"), col("F_score"), col("M_score"))
)

# Show only final columns
rfm_metrics.select(
    "Customer_ID",
    "Recency",
    "Frequency",
    "Monetary",
    "R_score",
    "F_score",
    "M_score",
    "RFM_score"
).orderBy("Customer_ID").show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+-------+---------+--------+-------+-------+-------+---------+
|Customer_ID|Recency|Frequency|Monetary|R_score|F_score|M_score|RFM_score|
+-----------+-------+---------+--------+-------+-------+-------+---------+
|      12346|    326|       12|77556.46|      2|      5|      5|      255|
|      12347|      3|        8| 4921.53|      5|      4|      5|      545|
|      12348|     76|        5|  2019.4|      3|      4|      4|      344|
|      12349|     19|        4| 4428.69|      5|      3|      5|      535|
|      12350|    311|        1|   334.4|      2|      1|      2|      212|
|      12351|    376|        1|  300.93|      2|      1|      2|      212|
|      12352|     37|       10| 2849.84|      4|      5|      4|      454|
|      12353|    205|        2|  406.76|      2|      2|      2|      222|
|      12354|    233|        1|  1079.4|      2|      1|      3|      213|
|      12355|    215|        2|  947.61|      2|      2|      3|      223|
|      12356|     23|    

In [0]:
# 12. Customer Segmentation

rfm_metrics = rfm_metrics.withColumn(
    "RF_score",
    concat(col("R_score"), col("F_score"))
)

rfm_metrics = rfm_metrics.withColumn(
    "Segment",
    when(col("RF_score").rlike("^[1-2][1-2]$"), "Hibernating")
    .when(col("RF_score").rlike("^[1-2][3-4]$"), "At Risk")
    .when(col("RF_score").rlike("^[1-2]5$"), "Can't Lose")
    .when(col("RF_score").rlike("^3[1-2]$"), "About to Sleep")
    .when(col("RF_score").rlike("^33$"), "Need Attention")
    .when(col("RF_score").rlike("^[3-4][4-5]$"), "Loyal Customers")
    .when(col("RF_score").rlike("^41$"), "Promising")
    .when(col("RF_score").rlike("^51$"), "New Customers")
    .when(col("RF_score").rlike("^[4-5][2-3]$"), "Potential Loyalists")
    .when(col("RF_score").rlike("^5[4-5]$"), "Champions")
)

rfm_metrics.select(
    "Customer_ID", "Recency", "Frequency", "Monetary", "RFM_score", "Segment"
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+-------+---------+--------+---------+--------------+
|Customer_ID|Recency|Frequency|Monetary|RFM_score|       Segment|
+-----------+-------+---------+--------+---------+--------------+
|      14095|    723|        1|    2.95|      121|   Hibernating|
|      16738|    298|        1|    3.75|      211|   Hibernating|
|      13788|    506|        1|    3.75|      111|   Hibernating|
|      14792|     64|        1|     6.2|      311|About to Sleep|
|      15913|    535|        1|     6.3|      121|   Hibernating|
|      15040|    542|        1|    7.49|      121|   Hibernating|
|      18115|    698|        1|     9.7|      121|   Hibernating|
|      17378|    375|        1|   10.95|      211|   Hibernating|
|      17956|    250|        1|   12.75|      211|   Hibernating|
|      16878|     85|        1|    13.3|      311|About to Sleep|
|      14900|    523|        1|   13.92|      111|   Hibernating|
|      14580|    519|        1|   14.85|      111|   Hibernating|
|      128

In [0]:
# 13. Segment Summary

segment_summary = (
    rfm_metrics
    .groupBy("Segment")
    .agg(
        round(avg("Recency")).alias("Recency_mean"),
        count("*").alias("Recency_count"),
        round(avg("Frequency")).alias("Frequency_mean"),
        count("*").alias("Frequency_count"),
        round(avg("Monetary")).alias("Monetary_mean"),
        count("*").alias("Monetary_count")
    )
)

segment_summary.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------+------------+-------------+--------------+---------------+-------------+--------------+
|            Segment|Recency_mean|Recency_count|Frequency_mean|Frequency_count|Monetary_mean|Monetary_count|
+-------------------+------------+-------------+--------------+---------------+-------------+--------------+
|      New Customers|        11.0|           71|           1.0|             71|        351.0|            71|
|          Promising|        38.0|          163|           1.0|            163|        335.0|           163|
|     About to Sleep|       108.0|          422|           1.0|            422|        533.0|           422|
|        Hibernating|       445.0|         1429|           1.0|           1429|        395.0|          1429|
|Potential Loyalists|        26.0|          711|           3.0|            711|       1220.0|           711|
|            At Risk|       407.0|          831|           4.0|            831|       1256.0|           831|
|     Need Attentio

In [0]:
# Revenue contribution by segment
segment_revenue = (
    rfm_metrics
    .groupBy("Segment")
    .agg(round(sum("Monetary"), 2).alias("total_revenue"))
)

total_revenue = segment_revenue.agg(sum("total_revenue")).collect()[0][0]

segment_revenue = segment_revenue.withColumn(
    "revenue_pct",
    round(col("total_revenue") / total_revenue * 100, 2)
)

segment_revenue.orderBy(col("total_revenue").desc()).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------+-------------+-----------+
|            Segment|total_revenue|revenue_pct|
+-------------------+-------------+-----------+
|          Champions|   8937954.57|      51.44|
|    Loyal Customers|   4660777.96|      26.82|
|            At Risk|   1043968.62|       6.01|
|Potential Loyalists|    867483.92|       4.99|
|         Can't Lose|    635833.47|       3.66|
|        Hibernating|     564879.3|       3.25|
|     Need Attention|    359575.81|       2.07|
|     About to Sleep|    224765.33|       1.29|
|          Promising|     54664.96|       0.31|
|      New Customers|     24900.31|       0.14|
+-------------------+-------------+-----------+

